# Comparison: Gaussian vs Uniform Field Monte Carlo

This notebook compares two field distributions for Monte Carlo photon propagation:
1. **Uniform field**: Photons uniformly distributed across aperture
2. **Gaussian field**: Photons sampled from Gaussian intensity distribution

We simulate propagation with 0 scattering to compare the focal plane distributions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../')

from monte_carlo.core import ApertureSimulator
from monte_carlo.gaussian_beam import GaussianBeamSimulator

## Simulation Parameters

In [ ]:
# Common parameters
n_photons = 50000
wavelength = 0.532  # microns (green light)
focal_length = 10000.0  # microns = 10mm
NA = 0.1
n_medium = 1.0
random_seed = 42

## 1. Uniform Field Simulation

In [ ]:
# Uniform field simulator
uniform_sim = ApertureSimulator(
    n_photons=n_photons,
    wavelength=wavelength,
    focal_length=focal_length,
    numerical_aperture=NA,
    n_medium=n_medium,
    random_seed=random_seed
)

# Run simulation
uniform_results = uniform_sim.propagate()
uniform_focal = uniform_results['focal_positions']

print(f"Uniform field - Aperture radius: {uniform_sim.aperture_radius:.3f} microns")
print(f"Uniform field - Focal plane shape: {uniform_focal[0].shape}")

## 2. Gaussian Field Simulation

In [ ]:
# Gaussian field simulator
gaussian_sim = GaussianBeamSimulator(
    n_photons=n_photons,
    wavelength=wavelength,
    numerical_aperture=NA,
    focal_length=focal_length,
    n_medium=n_medium,
    z_focus=focal_length,  # Put focus at z=focal_length to match uniform sim
    truncation_coeff=4.0,  # Untruncated Gaussian beam
    random_seed=random_seed
)

# Run simulation
gaussian_results = gaussian_sim.simulate()
gaussian_focal = gaussian_results['final_positions']

print(f"Gaussian field - Aperture radius: {gaussian_sim.aperture_radius:.3f} microns")
print(f"Gaussian field - Beam waist w0: {gaussian_sim.w0:.3f} microns")
print(f"Gaussian field - Rayleigh range: {gaussian_sim.z_R:.3f} microns")
print(f"Gaussian field - Focal plane shape: {gaussian_focal[0].shape}")

## 3. Comparison: Focal Plane Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Calculate radial distances
uniform_r = np.sqrt(uniform_focal[0]**2 + uniform_focal[1]**2)
gaussian_r = np.sqrt(gaussian_focal[0]**2 + gaussian_focal[1]**2)

# 2D scatter plots
axes[0, 0].scatter(uniform_focal[0], uniform_focal[1], alpha=0.1, s=1)
axes[0, 0].set_title('Uniform Field - Focal Plane (2D)')
axes[0, 0].set_xlabel('x (microns)')
axes[0, 0].set_ylabel('y (microns)')
axes[0, 0].axis('equal')

axes[1, 0].scatter(gaussian_focal[0], gaussian_focal[1], alpha=0.1, s=1, color='orange')
axes[1, 0].set_title('Gaussian Field - Focal Plane (2D)')
axes[1, 0].set_xlabel('x (microns)')
axes[1, 0].set_ylabel('y (microns)')
axes[1, 0].axis('equal')

# 2D histograms
bins = 100
extent_uniform = [
    uniform_focal[0].min(), uniform_focal[0].max(),
    uniform_focal[1].min(), uniform_focal[1].max()
]
extent_gaussian = [
    gaussian_focal[0].min(), gaussian_focal[0].max(),
    gaussian_focal[1].min(), gaussian_focal[1].max()
]

H_uniform, xedges, yedges = np.histogram2d(
    uniform_focal[0], uniform_focal[1], bins=bins
)
im1 = axes[0, 1].imshow(
    H_uniform.T, origin='lower', extent=extent_uniform, cmap='Blues', aspect='auto'
)
axes[0, 1].set_title('Uniform Field - Intensity (2D)')
axes[0, 1].set_xlabel('x (microns)')
axes[0, 1].set_ylabel('y (microns)')
plt.colorbar(im1, ax=axes[0, 1])

H_gaussian, xedges, yedges = np.histogram2d(
    gaussian_focal[0], gaussian_focal[1], bins=bins
)
im2 = axes[1, 1].imshow(
    H_gaussian.T, origin='lower', extent=extent_gaussian, cmap='Oranges', aspect='auto'
)
axes[1, 1].set_title('Gaussian Field - Intensity (2D)')
axes[1, 1].set_xlabel('x (microns)')
axes[1, 1].set_ylabel('y (microns)')
plt.colorbar(im2, ax=axes[1, 1])

# Radial profiles
bins_radial = np.linspace(0, max(uniform_r.max(), gaussian_r.max()), 50)
hist_uniform, edges = np.histogram(uniform_r, bins=bins_radial)
hist_gaussian, _ = np.histogram(gaussian_r, bins=bins_radial)
bin_centers = (edges[:-1] + edges[1:]) / 2

axes[0, 2].plot(bin_centers, hist_uniform, 'b-', label='Uniform')
axes[0, 2].set_title('Uniform Field - Radial Profile')
axes[0, 2].set_xlabel('Radial distance (microns)')
axes[0, 2].set_ylabel('Photon count')
axes[0, 2].grid(True, alpha=0.3)

axes[1, 2].plot(bin_centers, hist_gaussian, 'orange', label='Gaussian')
axes[1, 2].set_title('Gaussian Field - Radial Profile')
axes[1, 2].set_xlabel('Radial distance (microns)')
axes[1, 2].set_ylabel('Photon count')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/gaussian_vs_uniform_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Quantitative Comparison

In [ ]:
print("="*60)
print("FOCAL PLANE STATISTICS")
print("="*60)

print("\nUniform Field:")
print(f"  Mean radius: {uniform_r.mean():.3f} microns")
print(f"  Std radius: {uniform_r.std():.3f} microns")
print(f"  Max radius: {uniform_r.max():.3f} microns")
print(f"  x range: [{uniform_focal[0].min():.3f}, {uniform_focal[0].max():.3f}]")
print(f"  y range: [{uniform_focal[1].min():.3f}, {uniform_focal[1].max():.3f}]")

print("\nGaussian Field:")
print(f"  Mean radius: {gaussian_r.mean():.3f} microns")
print(f"  Std radius: {gaussian_r.std():.3f} microns")
print(f"  Max radius: {gaussian_r.max():.3f} microns")
print(f"  x range: [{gaussian_focal[0].min():.3f}, {gaussian_focal[0].max():.3f}]")
print(f"  y range: [{gaussian_focal[1].min():.3f}, {gaussian_focal[1].max():.3f}]")

print("\n" + "="*60)
print("KEY DIFFERENCES")
print("="*60)
print(f"Gaussian beam is more concentrated (smaller mean radius)")
print(f"Gaussian has smoother, natural intensity falloff")
print(f"Uniform has sharper cutoff (Airy pattern structure)")

## 5. Lens Plane Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

uniform_aperture = uniform_results['aperture_positions']
gaussian_lens = gaussian_results['lens_positions']

axes[0].scatter(uniform_aperture[0], uniform_aperture[1], alpha=0.1, s=1)
axes[0].set_title('Uniform Field - Aperture/Lens Plane')
axes[0].set_xlabel('x (microns)')
axes[0].set_ylabel('y (microns)')
axes[0].axis('equal')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(gaussian_lens[0], gaussian_lens[1], alpha=0.1, s=1, color='orange')
axes[1].set_title('Gaussian Field - Lens Plane')
axes[1].set_xlabel('x (microns)')
axes[1].set_ylabel('y (microns)')
axes[1].axis('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Uniform: Sharp cutoff at aperture edge (uniform within circle)")
print("Gaussian: Smooth Gaussian distribution truncated by aperture")